# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata object
dataset = mlc.Dataset(croissant_url)
# Print basic metadata information
meta_json = dataset.metadata.to_json()
print(f"Dataset Title: {meta_json['name']}")
print(f"Description: {meta_json['description']}")
print(f"Published: {meta_json.get('datePublished', 'N/A')}")
print(f"Identifier: {meta_json.get('identifier', 'N/A')}")
print(f"Croissant Version: {meta_json.get('conformsTo', 'N/A')}")
print("\nKeywords:")
print(', '.join(meta_json.get('keywords', [])))

## 2. Data Overview
Review available record sets, fields, and their IDs using the dataset Croissant schema.

In [ ]:
# List Record Sets and their @id
print("Record sets available in this dataset:")
record_sets = [rs['@id'] for rs in dataset.metadata.get('recordSet', [])]
for rs in dataset.metadata.get('recordSet', []):
    print(f"- Record Set Name: {rs.get('name','')}\n  @id: {rs['@id']}")
    # List fields in the record set
    if 'field' in rs:
        print(f"  Fields/Columns (@id):")
        for field in rs['field']:
            print(f"    - {field.get('@id', field)}: {field.get('name', '')}")
    print()
if not record_sets:
    print("No recordSet objects available in the metadata. Attempting to retrieve from schema manually...")
    # Fallback: try extracting from raw schema (for some Croissant schemas)
    schema_json = dataset.metadata.to_json()
    if 'recordSet' in schema_json:
        record_sets = [rs['@id'] for rs in schema_json['recordSet']]
        for rs in schema_json['recordSet']:
            print(f"- Record Set Name: {rs.get('name','')}\n  @id: {rs['@id']}")
            if 'field' in rs:
                print(f"  Fields/Columns (@id):")
                for field in rs['field']:
                    print(f"    - {field.get('@id', field)}: {field.get('name', '')}")
            print()
    else:
        print("No record sets found in schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this dataset, we'll choose the 
- record set: `https://api.app.sen.science/frontiers/7154140/d7635f4b-40d2-4130-8f38-2b1f876a021f` (main data table)

Let's inspect the first few records and look at the column names using their `@id` values.

In [ ]:
# Specify the record set (main data table in this dataset)
# Replace with the actual record set @id found above if differs!
MAIN_RECORD_SET_ID = 'https://api.app.sen.science/frontiers/7154140/d7635f4b-40d2-4130-8f38-2b1f876a021f'
record_sets = [MAIN_RECORD_SET_ID]

# Load each record set as a DataFrame
dataframes = {}
for rs_id in record_sets:
    print(f'Loading data from record set: {rs_id}')
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    print(f"Loaded {len(df)} records.")
    dataframes[rs_id] = df

# List the columns (these are the field @id's)
print("Available columns (field @id's):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

For demonstration, let's:
- Select the `cr:peptide_count` column (number of peptides detected, a numeric field).
- Filter for entries where `cr:peptide_count > 5`.
- Normalize `cr:peptide_count`.
- Optionally, group by the protein `cr:accession` (or similar).

<br>*(Adjust the field @id to match actual available columns in the DataFrame from above step if needed!)*

In [ ]:
# Set the @id for desired numeric and group fields (must match field ids from step above)
record_set_id = MAIN_RECORD_SET_ID
numeric_field = 'cr:peptide_count'  # change if actual @id differs
group_field = 'cr:accession'        # group by protein accession, change if actual @id differs

df = dataframes[record_set_id]

# Filter records: Only those with > 5 peptides
threshold = 5
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Records with {numeric_field} > {threshold}: {len(filtered_df)}")
else:
    raise KeyError(f"Field '{numeric_field}' not found in columns: {df.columns.tolist()}")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' (Z-score):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by accession to get mean statistics
if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped by {group_field}, mean {numeric_field}:")
    print(grouped_df.head())
else:
    print(f"Field '{group_field}' not in columns. Skipping groupby.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

- Histogram of `cr:peptide_count`.
- Boxplot of normalized `cr:peptide_count`.
- (Optional) Scatterplot of peptide count vs. MW if `cr:mw_kDa` is present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of peptide count
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title('Distribution of Peptide Counts')
plt.xlabel('Peptide Count')
plt.ylabel('Protein Entries')
plt.show()

# Boxplot of normalized peptide counts
plt.figure(figsize=(6, 5))
sns.boxplot(y=filtered_df[f'{numeric_field}_normalized'])
plt.title('Normalized Peptide Counts (Z-score)')
plt.show()

# Optional: Scatterplot of Peptide Count vs MW if available
mw_field = 'cr:mw_kDa'  # check available field @ids
if mw_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x=mw_field, y=numeric_field)
    plt.title('Peptide Count vs MW (kDa)')
    plt.xlabel('Molecular Weight [kDa]')
    plt.ylabel('Peptide Count')
    plt.show()
else:
    print(f"Field '{mw_field}' not in columns. Skipping MW scatterplot.")

## 6. Conclusion
In this notebook, we loaded the FAIR<sup>2</sup> mass spectrometry extracellular vesicle dataset using `mlcroissant`, inspected its structure using unique `@id` references, extracted and processed the main protein recordset, filtered and normalized peptide counts, and visualized the data distribution. 

- Most proteins have lower peptide counts, with a long tail at higher counts (see histogram).
- The normalization helps identify outlier proteins with unusually high representation.
- If more annotation/documentation fields are present, explore other record sets with similar techniques.  

You can extend this analysis by joining across record sets, exploring modification types, or relating abundance to other experimental parameters as annotated in the schema.